# PropRAG vs GraphRAG vs BaseRAG — Colab (free T4)

Runs the same `benchmark/` code as the desktop setup, but serves `gpt-oss-20b-Q4_K_M.gguf`
**in-process** via `llama-cpp-python` (CUDA build) instead of Koboldcpp — no localhost
server, no quantize-on-load.

**Why GGUF here, not native `transformers`**: `openai/gpt-oss-20b`'s native format uses
MXFP4 quantization, which requires GPU compute capability ≥ 9.0 (Hopper/Blackwell). The
free T4 is 7.5 — native format will not load. Q4_K_M GGUF (~11.6 GB) is the only way to
run this exact model on a T4.

**Before running**: zip `proprag_poc/` and `PropRAG-main/reproduce/dataset/*.json`
(preserving their relative paths, i.e. the zip's top level has `proprag_poc/` and
`PropRAG-main/` folders) and upload to Google Drive. Neither is on GitHub, so Colab can't
clone them — only this benchmark repo (`PropRAG Testing`) is. Set `DRIVE_ZIP_PATH` below.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## 1. Pull in `proprag_poc` + the 2wiki dataset (Drive) and clone this repo (GitHub)

In [ ]:
import os

DRIVE_ZIP_PATH = "/content/drive/MyDrive/PropRAG_local.zip"  # edit if you named it differently
PROPRAG_ROOT = "/content/PropRAG"

from google.colab import drive
drive.mount("/content/drive")

os.makedirs(PROPRAG_ROOT, exist_ok=True)
if not os.path.isfile(os.path.join(PROPRAG_ROOT, "proprag_poc", "config.py")):
    assert os.path.isfile(DRIVE_ZIP_PATH), (
        f"{DRIVE_ZIP_PATH} not found. Zip proprag_poc/ + "
        "PropRAG-main/reproduce/dataset/*.json (relative paths preserved) and upload it there, "
        "or edit DRIVE_ZIP_PATH above."
    )
    !unzip -q -o "{DRIVE_ZIP_PATH}" -d "{PROPRAG_ROOT}"
print("proprag_poc + dataset present:", os.path.isfile(os.path.join(PROPRAG_ROOT, "proprag_poc", "config.py")))

In [ ]:
import os

PROJECT_DIR = os.path.join(PROPRAG_ROOT, "PropRAG Testing")
if not os.path.isdir(PROJECT_DIR):
    !git clone https://github.com/ArnavJasoos/proprag_testing.git "{PROJECT_DIR}"
else:
    !cd "{PROJECT_DIR}" && git pull

# Layout must match the desktop setup: PROPRAG_ROOT/{proprag_poc, PropRAG-main, PropRAG Testing}
assert os.path.isfile(os.path.join(PROPRAG_ROOT, "proprag_poc", "config.py")), "proprag_poc missing"
assert os.path.isfile(os.path.join(PROPRAG_ROOT, "PropRAG-main", "reproduce", "dataset", "2wikimultihopqa.json")), "dataset missing"
assert os.path.isfile(os.path.join(PROPRAG_ROOT, "PropRAG-main", "reproduce", "dataset", "2wikimultihopqa_corpus.json")), "corpus missing"
assert os.path.isfile(os.path.join(PROJECT_DIR, "benchmark", "run.py")), "PropRAG Testing clone missing benchmark/run.py"
print("Layout OK:", PROJECT_DIR)

## 2. Install dependencies

`llama-cpp-python` is force-reinstalled with CUDA build flags afterward — the plain
`pip install` from `requirements.txt` gives a CPU-only build, too slow for a 20B model.
The CUDA build takes ~5–8 minutes to compile.

In [ ]:
!pip install -q -r "{PROJECT_DIR}/requirements.txt"

In [ ]:
%env CMAKE_ARGS=-DGGML_CUDA=on -DLLAMA_CUDA=on
%env FORCE_CMAKE=1
!pip install -q --no-cache-dir --force-reinstall llama-cpp-python

## 3. Configuration

In [ ]:
from getpass import getpass

HF_TOKEN = getpass("HF token (blank if the repo is public / you're not rate-limited): ") or None

GGUF_REPO = "unsloth/gpt-oss-20b-GGUF"
GGUF_FILENAME = "gpt-oss-20b-Q4_K_M.gguf"

N_QUESTIONS = 50
SEED = 42
PILOT = 10           # set to None for a full run
SYSTEMS = ["BaseRAG", "GraphRAG", "PropRAG"]
FORCE_REINDEX = False
MAKE_CHARTS = True

N_CTX = 4096          # conservative for T4's 15 GB; raise if you have more headroom
N_GPU_LAYERS = -1     # -1 = offload all layers; lower this if you hit CUDA OOM
N_BATCH = 512
N_THREADS = 2

## 4. Download the GGUF model (~11.6 GB, one-time per Colab runtime)

In [ ]:
from huggingface_hub import hf_hub_download

GGUF_MODEL_PATH = hf_hub_download(
    repo_id=GGUF_REPO, filename=GGUF_FILENAME, token=HF_TOKEN,
)
print("GGUF ready at", GGUF_MODEL_PATH)

## 5. Wire up the environment for `benchmark/`

In [ ]:
import os

os.environ["PROPRAG_MAIN"] = PROPRAG_ROOT
os.environ["PROPRAG_LLM_BACKEND"] = "llama_cpp"
os.environ["PROPRAG_GGUF_MODEL_PATH"] = GGUF_MODEL_PATH
os.environ["PROPRAG_LLAMA_N_CTX"] = str(N_CTX)
os.environ["PROPRAG_LLAMA_N_GPU_LAYERS"] = str(N_GPU_LAYERS)
os.environ["PROPRAG_LLAMA_N_BATCH"] = str(N_BATCH)
os.environ["PROPRAG_LLAMA_N_THREADS"] = str(N_THREADS)
# bge-large embeddings run on GPU automatically if sentence-transformers detects CUDA.

os.chdir(PROJECT_DIR)
print("cwd:", os.getcwd())

## 6. Smoke test

Loads the GGUF (first call only, can take 1–2 min), exercises all three indexes on 4 tiny
docs + 2 questions. This is the gate before spending time on a real run — it catches JSON
parse failures and gpt-oss reasoning leakage immediately.

In [ ]:
!python -m benchmark.smoke

## 7. Pilot run

Indexes the real 50-question subset's corpus, then runs the `PILOT`-sized stratified
subset through all three systems. Prints measured s/call and a projected full-run time.

In [ ]:
systems_arg = ",".join(SYSTEMS)
force_flag = "--force-reindex" if FORCE_REINDEX else ""
charts_flag = "" if MAKE_CHARTS else "--no-charts"
!python -m benchmark.run --questions {N_QUESTIONS} --pilot {PILOT} --seed {SEED} \
    --systems {systems_arg} {force_flag} {charts_flag}

## 8. Full run

Set `RUN_FULL = True` once the pilot's `report.md` and projected time look right. This can
run for hours; Colab free-tier sessions disconnect after ~12h (sooner if idle), and every
call is cached, so a disconnect just means rerunning this cell resumes from where it
left off (SQLite cache / stage checkpoints / `results.jsonl` all persist under `data/` —
copy that folder to Drive between sessions, see the last cell).

In [ ]:
RUN_FULL = False

if RUN_FULL:
    systems_arg = ",".join(SYSTEMS)
    charts_flag = "" if MAKE_CHARTS else "--no-charts"
    !python -m benchmark.run --questions {N_QUESTIONS} --seed {SEED} --systems {systems_arg} {charts_flag}
else:
    print("RUN_FULL is False - set it to True above and rerun this cell for the full 50-question run.")

## 9. View the report

In [ ]:
import glob, os
from IPython.display import Markdown, Image, display

run_dirs = sorted(glob.glob(os.path.join(PROJECT_DIR, "data", "benchmark", "*")), key=os.path.getmtime)
assert run_dirs, "no run directories found under data/benchmark yet"
latest_run_dir = run_dirs[-1]
print("Latest run:", latest_run_dir)

report_path = os.path.join(latest_run_dir, "report.md")
with open(report_path, encoding="utf-8") as f:
    display(Markdown(f.read()))

for chart in sorted(glob.glob(os.path.join(latest_run_dir, "charts", "*.png"))):
    display(Image(filename=chart))

## 10. Persist results to Drive

Colab's local disk is wiped when the runtime recycles. Copy `data/` (caches + all run
outputs) back to Drive so a disconnect doesn't lose cached LLM calls or results.

In [ ]:
import shutil, os

DRIVE_BACKUP_DIR = "/content/drive/MyDrive/PropRAG_benchmark_data"
src = os.path.join(PROJECT_DIR, "data")
if os.path.isdir(src):
    shutil.copytree(src, DRIVE_BACKUP_DIR, dirs_exist_ok=True)
    print("Backed up to", DRIVE_BACKUP_DIR)
else:
    print("No data/ directory yet - nothing to back up.")